# Forward renderer profile

This notebook measures the renderer users run in notebooks and documentation: early-exit sphere tracing, finite-difference normals, visibility sampling, and cached JIT programs. It deliberately does not benchmark image gradients because rendered pixels are no longer an optimization interface.

In [ ]:
from time import perf_counter

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from jaxcad.render import Camera, Material, RenderSettings, Scene, render_scene
from jaxcad.render.raymarch import _sphere_trace
from jaxcad.sdf.boolean import Union
from jaxcad.sdf.primitives import Plane, Sphere
from jaxcad.sdf.transforms import Translate

In [ ]:
geometry = Union(
    Translate(
        Sphere(0.9, material=Material(color=[0.8, 0.1, 0.05], roughness=0.5)), [-1.0, 0.0, 0.0]
    ),
    Translate(
        Sphere(0.9, material=Material(color=[0.9, 0.55, 0.08], roughness=0.18, metallic=0.9)),
        [1.0, 0.0, 0.0],
    ),
    Plane(-0.9, material=Material(color=[0.2, 0.23, 0.28], roughness=0.8)),
    smoothness=0.0,
)
scene = Scene(
    geometry,
    camera=Camera(position=(4.5, 2.8, 7.0), target=(0.0, -0.1, 0.0)),
    light_directions=((0.6, 1.0, 0.4), (-0.5, 0.4, -0.2)),
    light_colors=((1.0, 0.9, 0.75), (0.25, 0.35, 0.6)),
)

## Early termination

A direct sphere hit should consume only a small fraction of the available step budget. Misses also stop at `max_distance` rather than evaluating the SDF for every configured step.

In [ ]:
def sphere_sdf(point):
    return jnp.linalg.norm(point) - 1.0


hit = _sphere_trace(
    sphere_sdf, jnp.array([0.0, 0.0, 5.0]), jnp.array([0.0, 0.0, -1.0]), max_steps=96
)
miss = _sphere_trace(
    sphere_sdf, jnp.array([0.0, 5.0, 0.0]), jnp.array([0.0, 0.0, 1.0]), max_steps=96
)
print(f"hit: distance={float(hit.distance):.3f}, steps={int(hit.steps)}, hit={bool(hit.hit)}")
print(f"miss: distance={float(miss.distance):.3f}, steps={int(miss.steps)}, hit={bool(miss.hit)}")

## Compile once, render repeatedly

The first call includes JAX compilation. Later calls with the same scene, image shape, and static feature set reuse the module-level compiled renderer.

In [ ]:
settings = RenderSettings.balanced((128, 160))
start = perf_counter()
render_scene(scene, settings)
compile_and_render_ms = (perf_counter() - start) * 1e3

samples = []
for _ in range(10):
    start = perf_counter()
    render_scene(scene, settings)
    samples.append((perf_counter() - start) * 1e3)

print(f"first call: {compile_and_render_ms:.1f} ms")
print(f"cached median: {np.median(samples):.2f} ms")

## Preset cost and output

Each preset makes its work explicit. `draft` uses direct lighting with a shorter trace budget, `balanced` enables shadows and geometric AO, and `high_quality` adds tighter precision and 3×3 supersampling. Each distinct static preset compiles once.

In [ ]:
presets = {
    "Draft": RenderSettings.draft((128, 160)),
    "Balanced": RenderSettings.balanced((128, 160)),
    "High quality": RenderSettings.high_quality((128, 160)),
}
images, steady_ms = {}, {}
for name, preset in presets.items():
    images[name] = render_scene(scene, preset)  # compile/warm up
    timings = []
    for _ in range(5):
        start = perf_counter()
        images[name] = render_scene(scene, preset)
        timings.append((perf_counter() - start) * 1e3)
    steady_ms[name] = np.median(timings)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, (name, image) in zip(axes, images.items()):
    axis.imshow(image)
    axis.set_title(f"{name}\n{steady_ms[name]:.1f} ms cached")
    axis.axis("off")
plt.tight_layout()